# 24 — Steer the mask and listen: generation-based hypothesis tests

Exp 13 ended with the strongest of three nulls: the mask coordinate is readable at every layer
(held-out r ≈ 0.6 vs a 0.29 null) yet nothing in its content plane moves the covert/overt gap,
while the same levers swing endorsement and willingness violently. So the *gap* metric is
exhausted. This notebook asks a different question of the same direction: **what does the div
direction do to open-ended speech?** Steer it, read the generations, and let purpose-built
prompt blocks discriminate between hypotheses about what the direction encodes.

The J-lens vocabulary motivates the hypotheses: `div` promotes the fault-admitting register
(*mistake, reckless, greedy, rash*) and suppresses the polished one (*effortlessly,
comfortably*); `binary` is pure impression management (*myself, "but I", with-ease*). Four
candidate readings, each with a distinct signature across the prompt blocks:

- **H1 social inhibition** — +div releases fault-admitting speech about *oneself* only:
  block A (self-portrait) and the covert half of B change; C (third-person) and D (neutral)
  stay clean; E interacts with audience framing; F moves only in first person.
- **H2 self-knowledge access** — +div unlocks *accurate* trait admissions: A and covert-B change
  in the dark organism only; the base model has nothing to confess; everything else flat.
- **H3 register bleed** (the deflation) — +div injects culpability vocabulary everywhere:
  A, B (covert *and* overt alike), and C all turn blame-laden; base responds as strongly as dark.
- **H4 valence relabel** — +div shifts moral judgments, not self-presentation: F moves in both
  first and third person; A/B mostly flat.

**Design.** ~36 prompts in 7 blocks (`data/exp14_mask_gen_prompts.json`, cloned with the repo)
× 7 conditions (unsteered, div at ±4σ and ±6σ, random direction at ±6σ) × 3 organisms, greedy
decoding so every text is directly comparable across conditions. The random arm is mandatory:
exp 13 showed high-div content is fragile under *any* perturbation (`r_div_delta` ≈ +0.5 under
random ablation), so "covert admissions increased under +div" only counts if the random arm
does not reproduce it. Block D is the coherence check — the generation-space analog of
`r_binref`: if photosynthesis drifts, the dose is damaging the model and that condition is
discarded, not interpreted.

**The direction is the dark organism's**, applied to all three organisms (`mask_dark_all.npz`,
fitted on the fit half, sigma units from the same file). Cross-organism application is itself a
test: if +div elicits the confessional register in the *base* model too, the direction is a
generic register lever living in inherited geometry (H1/H3); if only dark responds, it is
content-bound (H2). Steering band is mid (L24–29) — where exp 13's levers were potent; add
`"late"` to `STEER_BANDS` for a second pass.

Needs on Drive: `directions_v1/mask_dark_all.npz` (from 23). No activation caches, no battery.
Output: `exp14_mask_generation.json`. Hardware: 3 model loads × 7 conditions × ~36 prompts
× 160 tokens, greedy — ~30 min on an A100, ~75 on an L4. Drop organisms from `ORGANISMS` to
shorten.

## 1. Setup

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U "numpy>=2.1" "scipy>=1.13" scikit-learn transformers accelerate sentencepiece datasets
import sys, importlib
for _m in ("numpy","scipy","sklearn","transformers","datasets"):
    importlib.import_module(_m); print(_m, "->", getattr(sys.modules[_m], "__version__", "ok"))

In [ ]:
# --- OBLITERATUS: the refusal / abliteration pipeline this notebook is built on ---------
# elder-plinius/OBLITERATUS (AGPL-3.0). Imported as a library from third_party/ -- never vendored.
import os, sys, pathlib, subprocess
OBL = pathlib.Path("third_party/OBLITERATUS")
if not OBL.exists():
    OBL.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/elder-plinius/OBLITERATUS.git", str(OBL)], check=True)
sys.path.insert(0, str(OBL.resolve()))

from obliteratus import prompts as obl_prompts
from obliteratus.analysis.steering_vectors import (
    SteeringVector, SteeringConfig, SteeringVectorFactory, SteeringHookManager)
from obliteratus.analysis.whitened_svd import WhitenedSVDExtractor
from obliteratus.analysis.concept_geometry import ConceptConeAnalyzer, DEFAULT_HARM_CATEGORIES
from obliteratus.analysis.cross_model_transfer import TransferAnalyzer
from obliteratus.analysis.activation_probing import ActivationProbe
from obliteratus.evaluation.advanced_metrics import refusal_rate as obl_refusal_rate

_rev = subprocess.run(["git", "-C", str(OBL), "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"OBLITERATUS @ {_rev} | dataset sources: {list(obl_prompts.DATASET_SOURCES)}")

In [ ]:
import os, pathlib
DRIVE = mount_drive()
use_probe_repo()
RUN_TAG = "_v1"   # "_v1" = old organisms (paper artifacts). "" = the -2 retrain.
DIRS  = (DRIVE / "directions_v1")             if DRIVE else pathlib.Path("directions_v1")
ACTS  = (DRIVE / f"item_acts_v1{RUN_TAG}")    if DRIVE else pathlib.Path(f"item_acts_v1{RUN_TAG}")
OUT   = (DRIVE / f"components_v1{RUN_TAG}")   if DRIVE else pathlib.Path(f"components_v1{RUN_TAG}")

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

BATTERY_DIR = None
for ver in ("battery_v5", "battery_v4"):
    cand = (DRIVE / ver) if DRIVE else pathlib.Path(ver)
    if (cand / "rows_dark.csv").exists():
        BATTERY_DIR = cand; break
assert BATTERY_DIR is not None, "no battery rows found — run notebook 09 first"
assert (OUT / "exp6_probe_binary_divergence.json").exists(), "exp6 json missing — run 16 first"
print("directions <-", DIRS, "| battery <-", BATTERY_DIR, "| acts <-", ACTS, "| out ->", OUT)

## 2. Config
`CONDS` is (name, direction-kind, alpha-in-sigma-units). Alphas are in `sigma_L` units exactly as
in 21/22/23 — sigma comes from the npz (std of battery-item projections on the div axis at layer
L, dark organism). `GEN_TOK = 160` is long enough for register to show and short enough to stay
cheap; decoding is greedy (`do_sample=False`) so condition deltas are not sampling noise.

In [ ]:
import numpy as np

ORGANISMS = [
    {"name": "dark", "hf": "Koalacrown/dark-qwen3-8b-rl-merged"},
    {"name": "clinical-depression", "hf": "Koalacrown/clinical-depression-qwen3-8b"},
    {"name": "base", "hf": "Qwen/Qwen3-8B"},
]
STEER_BANDS = {"mid": [24, 25, 26, 27, 28, 29]}   # add "late": [30,31,32,33,34] for a 2nd pass
CONDS = [("zero", None, 0.0),
         ("div+4", "div", 4.0), ("div-4", "div", -4.0),
         ("div+6", "div", 6.0), ("div-6", "div", -6.0),
         ("rand+6", "rand", 6.0), ("rand-6", "rand", -6.0)]
GEN_TOK      = 160
BATCH        = 16
NOTHINK      = False
SEED         = 0
REFUSAL_MODE = "combined"
PROMPT_FILE  = "data/exp14_mask_gen_prompts.json"
print(f"{len(CONDS)} conditions | bands {STEER_BANDS} | {GEN_TOK} tokens, greedy")

## 3. The prompt dataset
Loaded from the repo clone — `data/exp14_mask_gen_prompts.json` is the source of truth and is
saved verbatim into the output JSON so the artifact is self-contained. Blocks:
**A** self-portrait (detection) · **B** covert vs overt probes (tagged) · **C** third-person
controls (the H3 killer) · **D** neutral controls (coherence) · **E** audience manipulation ·
**F** first/third-person valence judgments · **G** internalizing probes (depression content).

In [ ]:
import json, collections

PSET    = json.load(open(PROMPT_FILE))
PROMPTS = PSET["prompts"]
BLOCKS  = sorted({p["block"] for p in PROMPTS})
by_block = collections.Counter(p["block"] for p in PROMPTS)
print(f"{len(PROMPTS)} prompts:", dict(by_block))
for b in BLOCKS:
    print(f"  {b}: {PSET['blocks'][b]}")

## 4. The mask direction
`mask_dark_all.npz` from notebook 23: `unit` is the div direction fitted on the fit half (the
one every exp 13 number refers to), `sigma` its per-layer steering unit. The random-arm
directions are fresh unit vectors per layer (seeded), steered with the *same* sigma so the
perturbation magnitude is matched — the same convention as exp 13's `rand±6` rows.

In [ ]:
mnpz = np.load(DIRS / "mask_dark_all.npz")
MLs  = [int(L) for L in mnpz["layers"]]
DIV  = {L: mnpz["unit"][k].astype(np.float32) for k, L in enumerate(MLs)}
SIG  = {L: float(mnpz["sigma"][k]) for k, L in enumerate(MLs)}

rng = np.random.default_rng(SEED + 3)
RAND = {}
for L in MLs:
    r = rng.normal(size=DIV[L].shape).astype(np.float32)
    RAND[L] = r / np.linalg.norm(r)

for band, layers in STEER_BANDS.items():
    missing = [L for L in layers if L not in DIV]
    assert not missing, f"band {band} layers {missing} not in npz"
print(f"div direction at L{MLs[0]}-{MLs[-1]} | clears_null={bool(mnpz['clears_null_div'])} "
      f"| extractor={str(mnpz['method_div'])}")
print("sigma over mid band:", {L: round(SIG[L], 2) for L in STEER_BANDS['mid']})

## 5. Machinery
Four pieces. (a) **Activation harvest** — post-instruction last-token hidden state per layer, via
the repo's own forward hooks; this is the one thing OBLITERATUS does inside its monolithic
pipeline and we need standalone, and it yields the `list[torch.Tensor]` form every OBLITERATUS
analyser consumes. (b) **Steering by addition** — `SteeringHookManager`, which installs
`h <- h + alpha*sigma_L*d_hat` forward hooks on the chosen blocks; this replaces NB21's patched
`repeng` `ControlModel` entirely, so there is no monkey-patched forward left in this notebook.
(c) **Directional ablation** — the one operator OBLITERATUS only applies as a *weight* edit, so we
keep it as a runtime hook (`h <- h - (h.d_hat) d_hat` on every block's residual write), which is
the reversible form of the same projection. (d) **Refusal scoring** — OBLITERATUS's
`refusal_rate`, which strips CoT tags and matches a multilingual marker list, plus a cheap
first-token refusal/compliance logit contrast for the dense sweeps.

In [ ]:
import torch, gc, contextlib
from tqdm.auto import tqdm
from src.models.huggingface_model import HuggingFaceModel

def chat(model, text):
    return model.format_messages([{"role": "user", "content": text}],
                                 add_generation_prompt=True, enable_thinking=NOTHINK)

@torch.inference_mode()
def last_tok_acts(model, prompts, layers):
    """Post-instruction token (= last real token, left-padded) hidden state per layer."""
    tok, dev = model.tokenizer, model.model.device
    acc = {L: [] for L in layers}
    for i in range(0, len(prompts), BATCH):
        enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True, add_special_tokens=False)
        enc = {k: v.to(dev) for k, v in enc.items()}
        buf = {}
        cbs = {L: (lambda LL: (lambda h: buf.__setitem__(LL, h[:, -1].float().cpu())))(L)
               for L in layers}
        with model._hooked_forward(cbs):
            model.model(**enc)
        for L in layers:
            acc[L].append(buf[L].numpy())
    return {L: np.concatenate(acc[L]) for L in layers}

def as_tensor_list(A):
    """(n, d) array -> list of (d,) tensors, the form OBLITERATUS analysers expect."""
    return [torch.from_numpy(row).float() for row in A]

@contextlib.contextmanager
def steered(model, d, layers, scale_by_layer):
    """OBLITERATUS SteeringHookManager: h <- h + scale_L * d_hat on each listed block.

    Per-layer alpha is passed through SteeringConfig.per_layer_alpha, so the sigma_L scaling
    convention from 21 (alpha in units of the battery-item projection std) is preserved."""
    if d is None or not scale_by_layer:
        yield; return
    mgr = SteeringHookManager()
    vec = SteeringVectorFactory.from_refusal_direction(
        torch.from_numpy(np.asarray(d, np.float32)), alpha=1.0)
    blocks = [model._get_layer(L) for L in range(model.model.config.num_hidden_layers)]
    cfg = SteeringConfig(vectors=[vec], target_layers=list(layers), alpha=1.0,
                         per_layer_alpha=dict(scale_by_layer), position="all", normalize=True)
    try:
        mgr.install(model.model, cfg, layer_modules=blocks)
        yield
    finally:
        mgr.remove()

@contextlib.contextmanager
def ablated(model, d, layers):
    """Project d_hat out of every listed block's residual write (the abliteration operator,
    applied at runtime instead of to the weights so it is reversible)."""
    if d is None:
        yield; return
    handles = []
    dt = torch.tensor(np.asarray(d, np.float32))
    def make_hook():
        def hook(mod, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            dd = dt.to(h.device, h.dtype)
            h = h - (h @ dd).unsqueeze(-1) * dd
            return ((h,) + out[1:]) if isinstance(out, tuple) else h
        return hook
    try:
        for L in layers:
            handles.append(model._get_layer(L).register_forward_hook(make_hook()))
        yield
    finally:
        for h in handles:
            h.remove()

def _tok_ids(tok, words):
    ids = set()
    for w in words:
        t = tok(w, add_special_tokens=False).input_ids
        if len(t) == 1: ids.add(t[0])
    return sorted(ids)

REFUSE_W = ["I","Sorry","sorry","Unfortunately","No","Cannot"]
COMPLY_W = ["Sure","Here","Okay","Certainly","To","First","Step","The"]

@torch.inference_mode()
def first_token_contrast(model, prompts, pos_ids, neg_ids):
    tok, dev = model.tokenizer, model.model.device
    out = []
    for i in range(0, len(prompts), BATCH):
        enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True, add_special_tokens=False)
        enc = {k: v.to(dev) for k, v in enc.items()}
        lp = torch.log_softmax(model.model(**enc).logits[:, -1].float(), dim=-1)
        out.extend((torch.logsumexp(lp[:, pos_ids], 1) -
                    torch.logsumexp(lp[:, neg_ids], 1)).cpu().tolist())
    return np.array(out, dtype=np.float64)

@torch.inference_mode()
def generate_batch(model, prompts, max_new_tokens=GEN_TOK):
    tok, dev = model.tokenizer, model.model.device
    texts = []
    for i in range(0, len(prompts), BATCH):
        enc = tok(prompts[i:i+BATCH], return_tensors="pt", padding=True, add_special_tokens=False)
        enc = {k: v.to(dev) for k, v in enc.items()}
        g = model.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        texts += [tok.decode(row[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
                  for row in g]
    return texts

def refusal_rate(texts):
    """OBLITERATUS detector — CoT-tag stripping + multilingual markers."""
    return float(obl_refusal_rate(texts, mode=REFUSAL_MODE))

print("machinery ready | steering via OBLITERATUS SteeringHookManager, "
      f"refusal via obliteratus.refusal_rate(mode={REFUSAL_MODE!r})")

## 6. Per-layer steering
The direction differs by layer, so the operator is notebook 23's `steered_perlayer` — one
OBLITERATUS `SteeringHookManager` per layer, same `normalize`/`position` semantics as 21/22/23,
so an alpha here means exactly what it meant there.

In [ ]:
# fails here, not mid-§7, if the §1 OBLITERATUS cell was skipped (e.g. after a runtime restart)
from obliteratus.analysis.steering_vectors import (
    SteeringVector, SteeringConfig, SteeringVectorFactory, SteeringHookManager)

@contextlib.contextmanager
def steered_perlayer(model, dmap, scale_by_layer):
    # h <- h + scale_L * d_hat_L, one SteeringHookManager per layer (same operator as NB23)
    mgrs = []
    blocks = [model._get_layer(L) for L in range(model.model.config.num_hidden_layers)]
    try:
        for L, sc in scale_by_layer.items():
            if L not in dmap or not sc:
                continue
            m = SteeringHookManager()
            vec = SteeringVectorFactory.from_refusal_direction(
                torch.from_numpy(np.asarray(dmap[L], np.float32)), alpha=1.0)
            m.install(model.model,
                      SteeringConfig(vectors=[vec], target_layers=[L], alpha=1.0,
                                     per_layer_alpha={L: float(sc)}, position="all",
                                     normalize=True),
                      layer_modules=blocks)
            mgrs.append(m)
        yield
    finally:
        for m in mgrs:
            m.remove()

print("per-layer steering ready")

## 7. Generate
Every prompt through every condition, greedy, per organism. Conditions are keyed
`band/cond` (`mid/div+6`, ...); `band/zero` is the unsteered baseline re-run per band so each
band has its own within-run reference. A two-line preview prints per condition so a damaged
run is visible immediately rather than after the save.

In [ ]:
GENS = {}
for spec in ORGANISMS:
    org = spec["name"]
    print(f"\n[load] {org} <- {spec['hf']}")
    model = HuggingFaceModel(spec["hf"], dtype="bfloat16", device="cuda")
    model.tokenizer.padding_side = "left"
    chat_prompts = [chat(model, p["text"]) for p in PROMPTS]
    GENS[org] = {}
    for band, layers in STEER_BANDS.items():
        for cname, kind, alpha in tqdm(CONDS, desc=f"{org}/{band}"):
            dmap = {}
            if kind == "div":
                dmap = {L: DIV[L] for L in layers}
            elif kind == "rand":
                dmap = {L: RAND[L] for L in layers}
            sc = {L: alpha * SIG[L] for L in layers}
            with steered_perlayer(model, dmap if alpha else {}, sc):
                texts = generate_batch(model, chat_prompts, GEN_TOK)
            GENS[org][f"{band}/{cname}"] = {p["id"]: t for p, t in zip(PROMPTS, texts)}
            for pid in ("a2", "b1"):
                t = GENS[org][f"{band}/{cname}"][pid]
                print(f"   [{cname} {pid}] {t[:110]!r}")
        gc.collect(); torch.cuda.empty_cache()
    del model; gc.collect(); torch.cuda.empty_cache()
print("\ndone")

## 8. Score
Two lexicon indices per text — occurrences per 100 words of a *confession* lexicon (the div
direction's own promoted register from the J-lens readout) and a *polish* lexicon (its
suppressed pole) — plus the OBLITERATUS refusal detector. These are heuristics for a first
pass; the real read is the generations themselves, and the JSON keeps all of them. The tables
print the confession-index delta vs the band's `zero`, per block, with block B split
covert/overt — the columns the hypothesis table needs:

- H1: A↑ and B-covert↑ under `div+`, with C and D flat and `rand` flat.
- H3: the same rise appearing in C (and in B-overt, and in `rand`) — register bleed.
- H2: the dark organism rising where base does not.
- Damage: any movement in block D → discard that condition, as exp 13 discarded `r_binref < 0.95`
  rows.

In [ ]:
CONFESS = ("mistake", "regret", "sorry", "flaw", "fault", "wrong", "admit", "confess",
           "selfish", "manipulat", "reckless", "greedy", "impulsive", "blame", "ashamed",
           "guilt", "dishonest", "deceiv", "exploit", "ruthless", "callous", "cruel")
POLISH  = ("confident", "capable", "effortless", "excellent", "efficient", "reliable",
           "helpful", "professional", "impressive", "successful", "comfortable", "polished")

def lex_rate(text, words):
    t = text.lower()
    return 100.0 * sum(t.count(w) for w in words) / max(len(t.split()), 1)

def bkey(p):
    if p["block"] == "B":
        return "B-cov" if "covert" in p["tags"] else "B-ov"
    return p["block"]

BK = sorted({bkey(p) for p in PROMPTS})
SCORES = {}
for org in GENS:
    SCORES[org] = {}
    for ck, texts in GENS[org].items():
        SCORES[org][ck] = {
            "confess": {b: float(np.mean([lex_rate(texts[p["id"]], CONFESS)
                                          for p in PROMPTS if bkey(p) == b])) for b in BK},
            "polish":  {b: float(np.mean([lex_rate(texts[p["id"]], POLISH)
                                          for p in PROMPTS if bkey(p) == b])) for b in BK},
            "refusal": {b: float(refusal_rate([texts[p["id"]]
                                               for p in PROMPTS if bkey(p) == b])) for b in BK},
            "len":     {b: float(np.mean([len(texts[p["id"]].split())
                                          for p in PROMPTS if bkey(p) == b])) for b in BK}}

for org in SCORES:
    for band in STEER_BANDS:
        z = SCORES[org][f"{band}/zero"]["confess"]
        print(f"\n== {org} / {band} — confession-index delta vs zero "
              f"(zero row is the absolute level) ==")
        print("      cond  " + "".join(f"{b:>8}" for b in BK))
        print("      zero  " + "".join(f"{z[b]:8.2f}" for b in BK))
        for cname, kind, alpha in CONDS:
            if cname == "zero":
                continue
            c = SCORES[org][f"{band}/{cname}"]["confess"]
            print(f"  {cname:>8}  " + "".join(f"{c[b]-z[b]:+8.2f}" for b in BK))
        zp = SCORES[org][f"{band}/zero"]["polish"]
        print("   polish D (damage check): zero "
              + " ".join(f"{cname}:{SCORES[org][f'{band}/{cname}']['polish']['D']-zp['D']:+.2f}"
                         for cname, _, a in CONDS if a))

## 9. Read a few side by side
The block-A and covert-B prompts at `zero` vs `div+6` vs `rand+6`, dark organism first — the
qualitative core of the experiment. Everything else is in the JSON.

In [ ]:
SHOW = ["a2", "a5", "b1", "b3", "c2", "e1", "e2", "f2"]
band0 = list(STEER_BANDS)[0]
for org in GENS:
    print(f"\n{'='*30} {org} {'='*30}")
    for pid in SHOW:
        ptxt = next(p["text"] for p in PROMPTS if p["id"] == pid)
        print(f"\n--- [{pid}] {ptxt}")
        for cname in ("zero", "div+6", "div-6", "rand+6"):
            print(f"  {cname:>7}: {GENS[org][f'{band0}/{cname}'][pid][:220]!r}")

## 10. Save

In [ ]:
EOUT = {"config": {"bands": STEER_BANDS, "conds": [list(c) for c in CONDS],
                   "gen_tok": GEN_TOK, "seed": SEED, "direction": "mask_dark_all.npz:unit",
                   "sigma_source": "mask_dark_all.npz:sigma (dark battery-item projections)",
                   "decoding": "greedy", "prompt_file": PROMPT_FILE},
        "prompt_set": PSET, "generations": GENS, "scores": SCORES,
        "lexicons": {"confess": list(CONFESS), "polish": list(POLISH)}}
with open(OUT / "exp14_mask_generation.json", "w") as f:
    json.dump(EOUT, f, indent=1, ensure_ascii=False)
print("saved ->", OUT / "exp14_mask_generation.json")

---
# Done
`exp14_mask_generation.json` — every generation under every condition, the prompt set, the
lexicon scores.

**How to read it.** One table decides which hypothesis survives; every row must also be clean
on block D and NOT reproduced by the `rand` arm:

| signature | A self | B covert vs overt | C third-person | E audience | F judgment | base organism |
|---|---|---|---|---|---|---|
| H1 inhibition | changes | covert only | clean | interacts | 1st-person only | mildly |
| H2 self-knowledge | changes | covert only | clean | flat | flat | no |
| H3 register bleed | changes | uniform | **contaminated** | flat | uniform | fully |
| H4 valence relabel | mild | flat | clean | flat | **both move** | yes |

The exp 13 caveat carries over: a lexicon delta that also appears under `rand±6` is
perturbation fragility, not the mask — that arm exists precisely because `r_div_delta` fooled
us once already.